A/B Test Design : Phase 6

In [1]:
from pathlib import Path
import math, duckdb, numpy as np, pandas as pd

root = Path.cwd().parent
proc = root / "data" / "processed"

con = duckdb.connect()
con.execute("SET memory_limit = '1.5GB'")
con.execute("SET threads = 2")
tmp = root / "data" / "duckdb_tmp"
tmp.mkdir(parents=True, exist_ok=True)
con.execute(f"SET temp_directory = '{tmp.as_posix()}'")

con.execute(f"CREATE OR REPLACE VIEW transactions AS SELECT * FROM '{(proc / 'transactions_train.parquet').as_posix()}'")
con.execute("""
    CREATE OR REPLACE VIEW tx AS
    SELECT *, CAST(date_diff('day', t_dat, (SELECT max(t_dat) FROM transactions)) // 7 AS INTEGER) AS week_ago
    FROM transactions
""")
con.sql("SELECT count(*) AS n FROM tx").df()

,n
0,31788324


Baseline: who enters the test, and how many come back within 30 days

In [2]:
def enrolment_cohort(week):
    """Customers who bought in `week`: did they buy again within 30 days? How often did they shop in the 90 days before?"""
    return con.sql(f"""
        WITH enrol AS (
            SELECT customer_id, min(t_dat) AS d0
            FROM tx WHERE week_ago = {week}
            GROUP BY 1
        )
        SELECT e.customer_id,
               max(CASE WHEN t.t_dat > e.d0 AND t.t_dat <= e.d0 + INTERVAL 30 DAY THEN 1 ELSE 0 END) AS repeat_30d,
               count(DISTINCT CASE WHEN t.t_dat < e.d0 AND t.t_dat >= e.d0 - INTERVAL 90 DAY THEN t.t_dat END) AS pre_90d_trips
        FROM enrol e
        JOIN tx t USING (customer_id)
        WHERE t.week_ago BETWEEN {week} - 5 AND {week} + 14
        GROUP BY 1
    """).df().assign(week=week)

cohorts = pd.concat([enrolment_cohort(w) for w in range(5, 13)], ignore_index=True)

weekly = cohorts.groupby("week").agg(customers=("customer_id", "size"), repeat_rate=("repeat_30d", "mean"))
display(weekly.round(4))

p0 = cohorts["repeat_30d"].mean()
print(f"Baseline 30-day repeat rate: {p0:.2%}  "
      f"(range {weekly['repeat_rate'].min():.2%} to {weekly['repeat_rate'].max():.2%})")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,customers,repeat_rate
week,,
5,71094,0.4806
6,74833,0.4838
7,82802,0.4901
8,80591,0.4880
9,77277,0.4914
10,75996,0.4958
11,85648,0.4909
12,115992,0.4886


Baseline 30-day repeat rate: 48.88%  (range 48.06% to 49.58%)


Sample Size for each min dectectable effect (MDE)

In [4]:
%pip install statsmodels

   ---------------------------------------- 0.0/11.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.4 MB 2.8 MB/s eta 0:00:04
   ---- ----------------------------------- 1.3/11.4 MB 3.2 MB/s eta 0:00:04
   ------ --------------------------------- 1.8/11.4 MB 3.1 MB/s eta 0:00:04
   ---------- ----------------------------- 2.9/11.4 MB 3.4 MB/s eta 0:00:03
   ----------- ---------------------------- 3.4/11.4 MB 3.5 MB/s eta 0:00:03
   -------------- ------------------------- 4.2/11.4 MB 3.5 MB/s eta 0:00:03
   ------------------ --------------------- 5.2/11.4 MB 3.5 MB/s eta 0:00:02
   ------------------ --------------------- 5.2/11.4 MB 3.5 MB/s eta 0:00:02
   ------------------- -------------------- 5.5/11.4 MB 3.2 MB/s eta 0:00:02
   --------------------- ------------------ 6.0/11.4 MB 2.8 MB/s eta 0:00:02
   ---------------------- ----------------- 6.3/11.4 MB 2.9 MB/s eta 0:00:02
   ----------------------- ---------------- 6.6/11.4 MB 2.6 MB/s eta 0:00:02
   ---


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

analysis = NormalIndPower()
rows = []
for mde in [0.02, 0.05, 0.10, 0.15]:
    p1 = p0 * (1 + mde)
    n = math.ceil(analysis.solve_power(effect_size=proportion_effectsize(p1, p0),
                                       alpha=0.05, power=0.80, ratio=1.0, alternative="two-sided"))
    rows.append({"relative_mde": f"{mde:.0%}",
                 "absolute_change_pp": round((p1 - p0) * 100, 2),
                 "customers_per_arm": n,
                 "total_customers": 2 * n})
plan = pd.DataFrame(rows)
plan

,relative_mde,absolute_change_pp,customers_per_arm,total_customers
0,2%,0.98,41062,82124
1,5%,2.44,6570,13140
2,10%,4.89,1641,3282
3,15%,7.33,727,1454


How Many weeks to enrol that many unique customers

In [6]:
cum = con.sql("""
    WITH last_week_seen AS (
        SELECT min(week_ago) AS w
        FROM tx WHERE week_ago >= 5
        GROUP BY customer_id
    )
    SELECT w - 4 AS weeks, sum(count(*)) OVER (ORDER BY w) AS unique_customers
    FROM last_week_seen GROUP BY w ORDER BY w
""").df()

def weeks_needed(total):
    hit = cum[cum["unique_customers"] >= total]
    return int(hit["weeks"].iloc[0]) if len(hit) else None

plan["enrolment_weeks"] = plan["total_customers"].apply(weeks_needed)
plan

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,relative_mde,absolute_change_pp,customers_per_arm,total_customers,enrolment_weeks
0,2%,0.98,41062,82124,2
1,5%,2.44,6570,13140,1
2,10%,4.89,1641,3282,1
3,15%,7.33,727,1454,1


Cuped and an A/A Check

In [7]:
rho = np.corrcoef(cohorts["repeat_30d"], cohorts["pre_90d_trips"])[0, 1]
vr = rho ** 2                                   # share of the metric's variance explained by pre-period shopping
n_plain = int(plan.loc[0, "customers_per_arm"])  # the 2% MDE row
n_cuped = math.ceil(n_plain * (1 - vr))

print(f"Correlation (pre-period trips vs 30-day repeat): {rho:.3f}")
print(f"Variance removed by CUPED: {vr:.1%}")
print(f"Customers per arm: {n_plain:,} -> {n_cuped:,} with CUPED")

Correlation (pre-period trips vs 30-day repeat): 0.322
Variance removed by CUPED: 10.4%
Customers per arm: 41,062 -> 36,793 with CUPED


A/A Check: Does the test false alarms?

In [8]:
from statsmodels.stats.proportion import proportions_ztest

rng = np.random.default_rng(42)
y = cohorts.loc[cohorts["week"] == 5, "repeat_30d"].to_numpy()

false_alarms = 0
for _ in range(1000):
    a = rng.random(len(y)) < 0.5                          # random 50/50 split: nobody gets a different treatment
    _, p = proportions_ztest([y[a].sum(), y[~a].sum()], [a.sum(), (~a).sum()])
    false_alarms += p < 0.05

print(f"A/A false-positive rate: {false_alarms / 1000:.1%} (should be close to 5%)")

A/A false-positive rate: 5.5% (should be close to 5%)


A/B Design Report

In [9]:
chosen = plan.loc[0]                                   # the 2% MDE row
fpr = false_alarms / 1000

mde_rows = "\n".join(
    f"| {r.relative_mde} | +{r.absolute_change_pp} | {r.customers_per_arm:,} | {r.total_customers:,} | {r.enrolment_weeks} |"
    for r in plan.itertuples()
)
week_rows = "\n".join(
    f"| {w} | {int(r.customers):,} | {r.repeat_rate:.2%} |" for w, r in weekly.iterrows()
)

report = f"""# A/B test design: personalised vs popularity recommendations

_Every number below is computed from the H&M data in `notebooks/06_ab_test.ipynb`._

| Item | Design |
|---|---|
| **Hypothesis** | Personalised post-purchase recommendations (LightGBM ranker) increase the 30-day repeat-purchase rate compared with popularity-based recommendations |
| **Unit and randomisation** | Customer; 50/50 split on a hash of `customer_id`, so each customer always sees the same arm |
| **Enrolment trigger** | A customer's first purchase during the enrolment period |
| **Primary metric** | Repeat purchase within 30 days of enrolment |
| **Secondary metrics** | Recommendation click-through rate, revenue per customer, items per basket |
| **Guardrails** | Return rate, API response time, catalogue coverage |
| **Baseline** | **{p0:.2%}** (enrolment weeks 5-12; weekly range {weekly['repeat_rate'].min():.2%} to {weekly['repeat_rate'].max():.2%}) |
| **Minimum detectable effect** | **{chosen.relative_mde} relative** (+{chosen.absolute_change_pp} points) |
| **Significance / power** | alpha = 0.05 (two-sided) / 80% |
| **Sample size** | **{n_cuped:,} customers per arm with CUPED** ({chosen.customers_per_arm:,} without) |
| **Duration** | **{chosen.enrolment_weeks} weeks of enrolment + 30 days of follow-up**, in whole weeks to cover every weekday |
| **Analysis** | Two-proportion z-test on the CUPED-adjusted metric |

## Baseline by week

| Weeks ago | Customers | 30-day repeat rate |
|---|---|---|
{week_rows}

## Sample size by minimum detectable effect

| Relative MDE | Absolute change (points) | Customers per arm | Total | Enrolment weeks |
|---|---|---|---|---|
{mde_rows}

Sample size grows with 1 / MDE squared. With about 80,000 buyers a week, traffic is not the constraint;
the 30-day follow-up sets the duration. The MDE is fixed before the test so it cannot be run "until significant".

## Variance reduction (CUPED)

Shopping days in the 90 days before enrolment correlate with the outcome at rho = {rho:.3f}.
CUPED removes rho squared = {vr:.1%} of the variance, cutting the sample from {chosen.customers_per_arm:,} to {n_cuped:,} per arm.

## A/A check

1,000 random 50/50 splits of one real week (identical treatment in both halves) gave a false-positive rate of
**{fpr:.1%}** against the expected 5%, so the test is well calibrated on this metric.

## Design decisions and risks

- **Population:** all weekly buyers (48.9% baseline) fill the test quickly. New customers only (14.8% month-1 retention)
  would target the biggest problem, since 27.8% never return, but would enrol far fewer people per week.
- **Novelty effect:** check the lift by week since enrolment before calling a winner.
- **Peeking:** analyse only at the planned end date.
- **Decision rule:** ship if the primary metric improves significantly and no guardrail gets worse beyond its threshold.

> This is an experiment **design**. A public dataset has no live traffic, so the test itself is not run.
"""

out = root / "reports" / "ab_test_design.md"
out.parent.mkdir(exist_ok=True)
out.write_text(report, encoding="utf-8")
print("saved", out)

saved c:\Users\simpl\projects\ecommerce-recommender\reports\ab_test_design.md
